## MMS Training
-----

This notebook tests the utilities in `src/train/mms_duration_experiment.py` for MMS adapter fine-tuning duration sweep experiments.

The workflow:
1. **Build processor once** from the full training dataset — vocabulary, tokenizer, and feature extractor are saved to disk and reused across all runs
2. **Fine-tune per duration** — load a fresh MMS model and train on progressively larger subsets (1h → 2h → 5h → 14h)
3. **Evaluate** on the held-out test set after each run to compare WER/CER across dataset sizes

> **Note:** The processor is built from the full 14h manifest so the vocabulary stays consistent across all duration experiments, making results directly comparable.



## 1.Setup

### 1.1 Python Imports

In [1]:
## Enviroment helper variable to help 
# when running the notebook in different environments (e.g. local vs colab)
ENV = "local"  # options: "local", "colab"

In [11]:
import sys, os
import time
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path

# Add project root (recommended)
if ENV == "jupyter-hub":
    PROJECT_SRC = Path("/content/drive/MyDrive/chichewa-asr/src")
else:
    PROJECT_SRC = Path().cwd().parents[1]

# Add project src to path to allow imports from src folder
sys.path.append(str(PROJECT_SRC))

# Import functions from src.train.train_whisper
from src.train.train_whisper import load_config

from src.train.mms_duration_experiment import (
    build_processor,
    load_model_and_processor,
    prepare_train_dataset,
    prepare_test_dataset,
    run_training,
    run_evaluation,
)


### 1.2. Input Folder and Other Configuration

In [3]:
# Base data directory
DIR_BASE = Path.cwd().parents[1]
DIR_DATA = DIR_BASE.joinpath('data')

# Direcotory for test data and manifest file
DIR_TEST = DIR_DATA / "test"
FILE_MANIFEST_TEST = DIR_TEST / "metadata.csv"

# Directory for dev data and manifest file
DIR_DEV = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"

# Directory for nested duration based data
DIR_DEV_NESTED_DURATION = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE / "configs" / "mms_hparams_debug.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "duration-exp-mms-1b-all"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)


# Model checkpoint directory (for saving model checkpoints during training)
DIR_MODELS = DIR_BASE / "models"
DIR_MODELS_ARTIFACTS = DIR_MODELS / "artifacts"


DIR_MODEL_CHECKPOINTS = DIR_MODELS / "checkpoints"
DIR_MODEL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)


In [4]:
# ==============================================
# TRAINING CONFIGURATION
# ==============================================
DEBUG = True  # Set to False for full training

### 1.3 Hugging Face Hub Log in

In [5]:
# Log into Hugging Face Hub (optional, required if you want to push the model to the hub)
if ENV == "colab":
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)

else:
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Experiment Configuration

In [6]:
# =========================================
# 1. LOAD HYPERPARAMETER CONFIG
# =========================================
config = load_config(FILE_CONFIG)


print(f"Config: {FILE_CONFIG.name}")
print(f"Model: {config['model']['model_name_or_path']}")


Config: mms_hparams_debug.yaml
Model: facebook/mms-1b-all


In [7]:
# ==============================================
# 2. SETUP MODEL-SPECIFIC ARTIFACT DIRECTORIES
# ==============================================
model_id = config["model"]["model_name_or_path"]
model_name = model_id.split("/")[-1]
target_lang = config["model"]["target_lang"]

# Create model-specific artifact directories
DIR_MODEL_ARTIFACT = DIR_MODELS_ARTIFACTS / model_name
DIR_PROCESSOR_ARTIFACT = DIR_MODEL_ARTIFACT / "processor"
DIR_VOCAB_ARTIFACT = DIR_MODEL_ARTIFACT / "vocab"

DIR_PROCESSOR_ARTIFACT.mkdir(parents=True, exist_ok=True)
DIR_VOCAB_ARTIFACT.mkdir(parents=True, exist_ok=True)

print(f"Model ID: {model_id}")
print(f"Processor dir: {DIR_PROCESSOR_ARTIFACT}")
print(f"Vocab dir: {DIR_VOCAB_ARTIFACT}")

Model ID: facebook/mms-1b-all
Processor dir: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/processor
Vocab dir: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/vocab


## 3. Load Model and Processor

In [8]:
# ==============================================
# 4.LOAD MODEL AND PROCESSOR
# ==============================================
model, processor = load_model_and_processor(config, 
                                            processor_dir=DIR_PROCESSOR_ARTIFACT)


  Loading processor from: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/processor
  Loading MMS model: facebook/mms-1b-all


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 25062.46it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([54])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([54, 1280])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Adapter loaded for lang: nya


### 4. Prepare Test Dataset

In [9]:
# ==============================================
# 1. PREPARE TEST DATASET
# ==============================================
dataset_test = prepare_test_dataset(
    manifest_path=FILE_MANIFEST_TEST,
    audio_dir=DIR_TEST,
    processor=processor,
    audio_fname_col="audio_filename",
    duration_col="duration_seconds",
)
print(f"Test set: {len(dataset_test):,} utterances")

  Loading test data: /Users/dmatekenya/git-repos/chichewa-asr/data/test/metadata.csv
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


Map (num_proc=1): 100%|██████████| 573/573 [00:10<00:00, 55.50 examples/s]

Test set: 573 utterances


## 5. Train Model for Multiple Files

In [10]:
# ================================
# BUILD DURATION_DATASETS 
# ===============================
DURATION_DATASETS = {
    f"{h}h": DIR_DEV_NESTED_DURATION / f"train_{h}h.csv"
    for h in range(1, 3)
}

# Sanity-check that all manifests exist before launching long runs
missing = [str(p) for p in DURATION_DATASETS.values() if not Path(p).exists()]
if missing:
    print("WARNING — manifest files not found:")
    for m in missing:
        print(f"  {m}")
else:
    print(f"All {len(DURATION_DATASETS)} manifest files found. Ready to run.")

All 2 manifest files found. Ready to run.


In [12]:
# ==========================================
# EXPERIMENT SETTINGS
# ==========================================
model_id     = config["model"]["model_name_or_path"]
model_name   = model_id.split("/")[-1]
HUB_MODEL_ID = f"dmatekenya/{model_name}-chichewa"

DEBUG = True  # set False for real runs on the server

summary = []

for duration_label, manifest_path in DURATION_DATASETS.items():
    if not manifest_path.exists():
        print(f"Skipping {duration_label} — manifest not found.")
        continue

    print(f"\n{'='*60}\n  EXPERIMENT: {duration_label}\n{'='*60}")

    hub_model_id = f"{HUB_MODEL_ID}-{duration_label}"
    output_dir   = DIR_MODEL_CHECKPOINTS / f"{model_name}-chichewa-{duration_label}"

    # 1. Load fresh model (processor is shared across all runs)
    model, processor = load_model_and_processor(config, processor_dir=DIR_PROCESSOR_ARTIFACT)

    # 2. Prepare training data
    dataset_train = prepare_train_dataset(manifest_path, DIR_DEV, processor)

    # 3. Train
    train_start = time.time()
    trainer = run_training(model, processor, dataset_train, config, hub_model_id, output_dir, debug=DEBUG)
    train_minutes = (time.time() - train_start) / 60

    # 4. Push to Hub (skipped in debug)
    if not DEBUG:
        print(f"  Pushing to Hub: {hub_model_id}")
        trainer.push_to_hub()

    # 5. Evaluate on held-out test set
    df_results = run_evaluation(model, processor, dataset_test, duration_label, DIR_RESULTS, model_id=hub_model_id, debug=DEBUG)
    summary.append({
        "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "duration":      duration_label,
        "wer":           df_results["wer_avg"].iloc[0],
        "cer":           df_results["cer_avg"].iloc[0],
        "hub_model_id":  hub_model_id,
        "train_minutes": round(train_minutes, 2),
    })

    # Save rolling summary so partial results survive a crash
    pd.DataFrame(summary).to_csv(DIR_RESULTS / "duration_sweep_summary.csv", index=False)

print("\nSweep complete.")
pd.DataFrame(summary)



  EXPERIMENT: 1h
  Loading processor from: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/processor
  Loading MMS model: facebook/mms-1b-all


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 29806.05it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([54])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([54, 1280])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Adapter loaded for lang: nya
  Loading train data: /Users/dmatekenya/git-repos/chichewa-asr/data/dev_nested_duration/train_1h.csv
Total duration : 1.00 hrs  (278 utterances)
  train       :   244 utterances  |  0.90 hrs  (89.9%)
  validation  :    34 utterances  |  0.10 hrs  (10.1%)


Map (num_proc=1): 100%|██████████| 34/34 [00:00<00:00, 38.10 examples/s]


  DEBUG: forcing CPU (MPS does not support CTC loss)
  Training ...


...

  DEBUG: forcing CPU (MPS does not support CTC loss)
  Training ...


Step,Training Loss,Validation Loss,Wer
10,36.223679,27.163477,1.000000
20,12.341367,16.152922,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


[DEBUG] Running evaluation on a small sample of the test set.
  WER (corpus): 100.00%   CER (corpus): 163.86%
  Predictions saved: /Users/dmatekenya/git-repos/chichewa-asr/outputs/duration-exp-mms-1b-all/predictions_2h.csv

Sweep complete.


,run_timestamp,duration,wer,cer,hub_model_id,train_minutes
0,2026-05-25 15:45,1h,100.0,165.781711,dmatekenya/mms-1b-all-chichewa-1h,2.62
1,2026-05-25 15:49,2h,100.0,163.864307,dmatekenya/mms-1b-all-chichewa-2h,3.65
